# Lab 02 · Manipulación de datos

*Análisis Avanzado de Datos con Python · Subsecretaría de Energía · Módulo 2*

Trabaja sobre tu propia copia del notebook. Todo lo que escribas queda en ella.

In [ ]:
#@title De qué se trata este lab { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#d9edf7;border-left:6px solid #31708f;color:#0b3d62"><div style="font-weight:700;font-size:16px;margin-bottom:6px">De qué se trata este lab</div><strong>Preguntas que vamos a responder</strong>
<ul>
<li>Cómo se le hace una pregunta distinta a la misma tabla sin volver a cargarla</li>
<li>Cuánto cambia la demanda entre un día hábil y un domingo</li>
<li>Qué hora del día genera cada tecnología, y por qué la solar y la hidro no se parecen</li>
<li>Cuánto se mueve el precio cuando se mueve la demanda</li>
</ul>
<strong>Al terminar vas a poder</strong>
<ul>
<li>Elegir filas y columnas con corchetes, <code>loc</code>, <code>iloc</code> y <code>query</code></li>
<li>Sacarle el mes, el día de la semana y la hora a una columna de fechas</li>
<li>Agrupar por una o varias claves y resumir con varios cálculos a la vez</li>
<li>Dar vuelta una tabla con <code>pivot_table</code> y deshacerla con <code>melt</code></li>
<li>Crear columnas calculadas, clasificar con cortes y repartir un total entre filas</li>
</ul></div>"""))

In [ ]:
#@title Datos del curso { display-mode: "form" }
#@markdown Corre esta celda. Deja listos los archivos del Observatorio de Datos Energeticos.
import numpy as np, pandas as pd, os, json, sqlite3
if not os.path.exists("centrales.csv"):
    rng = np.random.default_rng(2026)
    centrales = pd.DataFrame([
     ("Central Rio Manso Alto","hidro","Biobio",420,2004),("Central Salto Verde","hidro","Los Lagos",310,1998),
     ("Central Aguas Claras","hidro","Biobio",180,2011),("Central Vega Azul","hidro","Los Lagos",95,2016),
     ("Central Tres Saltos","hidro","Biobio",260,1995),
     ("Parque Solar Pampa Alta","solar","Antofagasta",230,2019),("Parque Solar Llano Seco","solar","Atacama",180,2020),
     ("Parque Solar Sol Naciente","solar","Antofagasta",145,2021),("Parque Solar Quebrada Honda","solar","Atacama",95,2022),
     ("Parque Solar Altiplano","solar","Antofagasta",310,2023),
     ("Eolica Cerro Negro","eolica","Coquimbo",160,2017),("Eolica Punta Ventosa","eolica","Coquimbo",120,2018),
     ("Eolica Loma Fria","eolica","Valparaiso",85,2020),("Eolica Campo Abierto","eolica","Coquimbo",200,2021),
     ("Termoelectrica Bahia Norte","gas","Valparaiso",375,2008),("Termoelectrica Puerto Sur","gas","Biobio",290,2012),
     ("Termoelectrica Valle Central","gas","Metropolitana",210,2006),
     ("Carboelectrica Costa Brava","carbon","Biobio",480,2001),("Carboelectrica Roca Gris","carbon","Antofagasta",350,1999),
     ("Diesel Respaldo Cordillera","diesel","Metropolitana",45,2014),
    ], columns=["central","tecnologia","region","potencia_mw","anio_inicio"])
    fechas = pd.date_range("2024-01-01","2024-12-31",freq="D")
    perfil = np.array([0,0,0,0,0,0,.05,.18,.38,.58,.75,.87,.93,.9,.8,.63,.42,.2,.05,0,0,0,0,0])
    filas=[]
    for _,c in centrales.iterrows():
        p,t = c.potencia_mw, c.tecnologia
        for f in fechas:
            est = 1+0.25*np.cos(2*np.pi*(f.dayofyear-15)/365)
            if t=="solar": base = p*perfil*0.30*est*rng.uniform(.8,1.1)
            elif t=="eolica": base = p*0.36*rng.uniform(.15,1.6,24)
            elif t=="hidro": base = p*0.55*(2-est)*rng.uniform(.9,1.1,24)
            elif t=="gas": base = p*0.68*rng.uniform(.9,1.05,24)
            elif t=="carbon": base = p*0.65*rng.uniform(.95,1.02,24)
            else:
                base = np.zeros(24); base[18:23] = p*0.55*rng.uniform(.8,1,5)
            filas.append(pd.DataFrame({"fecha":f.strftime("%Y-%m-%d"),"hora":range(24),
                                       "central":c.central,"mwh":np.clip(base,0,p).round(2)}))
    centrales.to_csv("centrales.csv", index=False)
    pd.concat(filas, ignore_index=True).to_csv("generacion.csv", index=False)

    # Excel con dos hojas, la segunda con notas en texto libre
    with pd.ExcelWriter("centrales.xlsx") as w:
        centrales.to_excel(w, sheet_name="centrales", index=False)
        pd.DataFrame({"nota":["Potencias declaradas al 31 de diciembre de 2024",
                              "Las centrales de pasada se informan con su potencia maxima"]}
                     ).to_excel(w, sheet_name="notas", index=False)

    # Demanda por region, base de datos SQLite
    regs = ["Antofagasta","Atacama","Coquimbo","Valparaiso","Metropolitana","Biobio","Los Lagos"]
    pobl = [700000,320000,850000,1900000,8100000,1700000,900000]
    perfil_d = np.array([.72,.68,.66,.65,.66,.70,.78,.88,.95,.98,1.0,1.02,1.03,1.0,.97,.96,.97,1.0,1.06,1.10,1.08,.98,.88,.79])
    dem=[]
    for r,p in zip(regs,pobl):
        base_r = p/8000
        for f in fechas:
            inv = 1+0.18*np.cos(2*np.pi*(f.dayofyear-190)/365)
            finde = 0.92 if f.dayofweek>=5 else 1.0
            v = base_r*perfil_d*inv*finde*rng.uniform(.97,1.03,24)
            dem.append(pd.DataFrame({"fecha":f.strftime("%Y-%m-%d"),"hora":range(24),
                                     "region":r,"mwh":v.round(2)}))
    demanda = pd.concat(dem, ignore_index=True)
    demanda.to_csv("demanda.csv", index=False)
    con = sqlite3.connect("demanda.db")
    demanda.to_sql("demanda", con, index=False, if_exists="replace")
    pd.DataFrame({"region":regs,"poblacion":pobl}).to_sql("regiones", con, index=False, if_exists="replace")
    con.close()

    # Precios de nudo de enero, como los entregaria una API REST
    ene = demanda[demanda["fecha"].str.startswith("2024-01")]
    pr = ene.assign(precio_usd_mwh=(40 + ene["mwh"]/ene["mwh"].max()*110
                                    + rng.normal(0,6,len(ene))).clip(40,180).round(2))
    json.dump({"metadata":{"fuente":"Observatorio de Datos Energeticos",
                           "fecha_consulta":"2024-02-01","unidad":"USD por MWh"},
               "datos": pr[["fecha","hora","region","precio_usd_mwh"]].to_dict("records")},
              open("precios_nudo.json","w"))
print("Datos listos")

In [ ]:
import pandas as pd

# El punto de partida de este lab es el punto de llegada del anterior.
generacion = pd.read_csv("generacion.csv", parse_dates=["fecha"])
centrales = pd.read_csv("centrales.csv")
demanda = pd.read_csv("demanda.csv", parse_dates=["fecha"])

# La tabla unida del Lab 01, generacion con la ficha de cada central pegada al lado.
gen = pd.merge(generacion, centrales, on="central", how="left")

print("gen    ", gen.shape, list(gen.columns))
print("demanda", demanda.shape, list(demanda.columns))

## 1. Elegir lo que importa

In [ ]:
# Los corchetes de afuera son "dame esto", los de adentro son la lista de columnas.
gen[["fecha", "hora", "central", "tecnologia", "mwh"]].head()

In [ ]:
# loc trabaja con nombres. Antes de la coma van las filas, despues las columnas.
# Ojo, loc incluye las dos puntas, asi que 0:4 son cinco filas.
gen.loc[0:4, ["central", "tecnologia", "mwh"]]

In [ ]:
# iloc trabaja con posiciones y deja fuera la ultima, asi que 0:4 son cuatro filas.
# Aca ademas pedimos las columnas 2, 5 y 6, en ese orden.
gen.iloc[0:4, [2, 5, 6]]

In [ ]:
# Una comparacion sobre una columna no devuelve un dato, devuelve una columna
# de verdaderos y falsos, una por fila.
es_solar = gen["tecnologia"] == "solar"

print(es_solar.head(3))
print(es_solar.sum(), "filas cumplen la condicion, de", len(gen))

In [ ]:
# Esa columna de verdaderos y falsos es la que se pasa entre corchetes.
solares = gen[es_solar]
print(solares.shape)

# Lo mismo escrito de corrido, que es como se ve en el codigo real.
print(gen[gen["tecnologia"] == "solar"].shape)

In [ ]:
# & es "y", | es "o", ~ es "no". Cada condicion va entre parentesis.
solar_norte = gen[(gen["tecnologia"] == "solar") & (gen["region"] == "Antofagasta")]
print("solar en Antofagasta      ", solar_norte.shape[0], "filas")

no_termicas = gen[~gen["tecnologia"].isin(["gas", "carbon", "diesel"])]
print("todo lo que no es termico ", no_termicas.shape[0], "filas")

In [ ]:
# isin pregunta si el valor esta en una lista y reemplaza a varios | seguidos.
sur = gen[gen["region"].isin(["Biobio", "Los Lagos"])]
print("regiones del sur", sur["region"].unique())

# between pregunta por un rango, con los dos extremos incluidos.
grandes = centrales[centrales["potencia_mw"].between(200, 400)]
print("centrales entre 200 y 400 MW", grandes.shape[0])

In [ ]:
# query recibe la condicion escrita como texto, sin corchetes ni parentesis.
print(gen.query("tecnologia == 'solar' and region == 'Antofagasta'").shape)

# El mismo resultado que el filtro de mas arriba, escrito de otra forma.

In [ ]:
#@title Tu turno { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Tu turno</div><strong>Cambia la pregunta, no el codigo</strong>
<p>La celda de abajo responde cuánta energía generó la solar de Antofagasta en enero. Cámbiala para responder otra, generación eólica o hidro, en Biobio o Los Lagos, durante el invierno, o sea entre el 1 de junio y el 31 de agosto. Pista, con <code>isin</code> resuelves las dos primeras condiciones.</p></div>"""))

In [ ]:
# Tu turno
respuesta = gen[
    (gen["tecnologia"] == "solar")
    & (gen["region"] == "Antofagasta")
    & (gen["fecha"].between("2024-01-01", "2024-01-31"))
]

print(respuesta.shape[0], "filas")
print(round(respuesta["mwh"].sum(), 1), "MWh")

## 2. Las fechas como herramienta

In [ ]:
# Todo esto sale de la misma columna fecha, sin cortar texto a mano.
print("anio       ", demanda["fecha"].dt.year.head(2).tolist())
print("mes        ", demanda["fecha"].dt.month.head(2).tolist())
print("dia        ", demanda["fecha"].dt.day_name().head(2).tolist())
print("dia numero ", demanda["fecha"].dt.dayofweek.head(2).tolist())

In [ ]:
# Cuatro columnas nuevas, calculadas para las 61.488 filas de una sola vez.
demanda["mes"] = demanda["fecha"].dt.month
demanda["nombre_mes"] = demanda["fecha"].dt.month_name()
demanda["dia_semana"] = demanda["fecha"].dt.dayofweek
demanda["es_finde"] = demanda["dia_semana"] >= 5

demanda.head(3)

In [ ]:
# Las dos que mas vamos a usar, ahora en la tabla de generacion.
gen["mes"] = gen["fecha"].dt.month
gen["es_finde"] = gen["fecha"].dt.dayofweek >= 5

print(gen["es_finde"].value_counts())

In [ ]:
# Demanda promedio por hora, separando dia habil de fin de semana.
print(demanda.groupby("es_finde")["mwh"].mean().round(1))

caida = demanda.groupby("es_finde")["mwh"].mean()
print("el fin de semana consume un", round(100 * (1 - caida[True] / caida[False]), 1),
      "por ciento menos")

In [ ]:
#@title Ojo con esta cifra { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#fdecea;border-left:6px solid #c0392b;color:#7b241c"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Ojo con esta cifra</div><p>El 7,8 por ciento de caída el fin de semana es de estos datos sintéticos.</p><p>En el sistema eléctrico real la caída existe, tanto que el Coordinador modela la demanda con cuatro días tipo, lunes, de martes a viernes, sábado y domingo. Pero no hay publicada una cifra oficial de cuánto cae, habría que calcularla con los datos horarios.</p></div>"""))

In [ ]:
# Una semana completa, comparando la columna de fechas contra texto.
semana = demanda[demanda["fecha"].between("2024-07-01", "2024-07-07")]

print("del", semana["fecha"].min().date(), "al", semana["fecha"].max().date())
print(semana.shape[0], "filas, que son 7 dias por 24 horas por 7 regiones")

In [ ]:
#@title Tu turno { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Tu turno</div><strong>Arma el trimestre</strong>
<p>El accesorio <code>dt</code> tiene <code>quarter</code>, que devuelve 1, 2, 3 o 4. Crea con él una columna <code>trimestre</code> en <code>demanda</code>, filtra el tercer trimestre y muestra cuántas filas quedaron.</p></div>"""))

In [ ]:
# Tu turno
# Cambia .dt.month por .dt.quarter y despues ajusta el filtro.
demanda["trimestre"] = demanda["fecha"].dt.month

seleccion = demanda[demanda["trimestre"] == 3]
print(seleccion.shape[0], "filas")

## 3. Agrupar y resumir en serio

In [ ]:
# Partir por tecnologia, sumar mwh en cada grupo, juntar los seis resultados.
gen.groupby("tecnologia")["mwh"].sum().round(0).sort_values(ascending=False)

In [ ]:
# Con dos claves, el resultado tiene una fila por combinacion que exista.
por_region = gen.groupby(["region", "tecnologia"])["mwh"].sum().round(0)

print(por_region.shape[0], "combinaciones de region y tecnologia")
por_region.head(6)

In [ ]:
# Con una lista de funciones se calculan todas sobre la misma columna.
gen.groupby("tecnologia")["mwh"].agg(["sum", "mean", "std", "max"]).round(1)

In [ ]:
# Con un diccionario, cada columna recibe el calculo que le corresponde.
gen.groupby("tecnologia").agg({"mwh": "sum", "potencia_mw": "mean",
                               "central": "nunique"}).round(1)

In [ ]:
# A la izquierda el nombre que quieres, a la derecha la columna y el calculo.
resumen = gen.groupby(["region", "tecnologia"]).agg(
    total=("mwh", "sum"),
    promedio=("mwh", "mean"),
    centrales=("central", "nunique"),
)

resumen.round(1).head()

In [ ]:
# reset_index devuelve las claves del groupby al cuerpo de la tabla.
resumen = resumen.reset_index()

print(resumen.columns.tolist())
resumen.sort_values("total", ascending=False).head(5).round(1)

In [ ]:
# size cuenta filas del grupo. count cuenta valores no nulos de cada columna.
print(gen.groupby("tecnologia").size())
print()
print(gen.groupby("tecnologia")["mwh"].count())

In [ ]:
#@title Tu turno { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Tu turno</div><strong>El mismo resumen, otra tabla</strong>
<p>Cambia la tabla por <code>demanda</code> y las claves por <code>region</code> y <code>dia_semana</code>. Pide el promedio y el máximo de <code>mwh</code> con agregación con nombre, y termina con <code>reset_index</code>.</p></div>"""))

In [ ]:
# Tu turno
# Cambia gen por demanda, las claves por region y dia_semana, y los calculos
# por promedio y maximo. Termina con reset_index.
resultado = gen.groupby(["region", "tecnologia"]).agg(
    total=("mwh", "sum"),
    promedio=("mwh", "mean"),
)

print(resultado.round(1).head())

## 4. Dar vuelta la tabla

In [ ]:
# index son las filas, columns son las columnas y values lo que se calcula.
perfil = gen.pivot_table(index="hora", columns="tecnologia",
                         values="mwh", aggfunc="mean")

perfil.round(1)

In [ ]:
# Exactamente el mismo calculo, presentado como tabla larga.
gen.groupby(["hora", "tecnologia"])["mwh"].mean().round(1).head(8)

In [ ]:
# fill_value llena los huecos y margins agrega la fila y la columna de totales.
tabla = gen.pivot_table(index="region", columns="tecnologia", values="mwh",
                        aggfunc="sum", fill_value=0, margins=True)

tabla.round(0)

In [ ]:
# Demanda promedio por region y mes, con los meses abriendose hacia el lado.
pd.set_option("display.width", 140)
mensual = demanda.pivot_table(index="region", columns="mes",
                              values="mwh", aggfunc="mean")

mensual.round(0)

In [ ]:
# melt deshace la pivot. Cada celda de la tabla ancha vuelve a ser una fila.
largo = mensual.reset_index().melt(id_vars="region", var_name="mes",
                                   value_name="mwh_promedio")

print(largo.shape, "que es 7 regiones por 12 meses")
largo.head(4)

In [ ]:
#@title Tu turno { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Tu turno</div><strong>Da vuelta la tabla</strong>
<p>Cambia <code>index</code> y <code>columns</code> para que las regiones queden en las filas y las tecnologías en las columnas, con la suma de <code>mwh</code>, y agrega los totales con <code>margins=True</code>.</p></div>"""))

In [ ]:
# Tu turno
# Cambia index por region, columns por tecnologia, aggfunc por suma, y agrega margins.
mi_pivot = gen.pivot_table(index="hora", columns="tecnologia",
                           values="mwh", aggfunc="mean")

mi_pivot.round(1).head()

## 5. Columnas que no venian en el archivo

In [ ]:
# Una division entre dos columnas se aplica a las 175.680 filas de una vez.
gen["factor_planta"] = gen["mwh"] / gen["potencia_mw"]

gen[["central", "mwh", "potencia_mw", "factor_planta"]].head(3).round(3)

In [ ]:
# El factor de planta promedio de cada tecnologia, ya con la columna creada.
gen.groupby("tecnologia")["factor_planta"].mean().round(3).sort_values(ascending=False)

In [ ]:
# La misma condicion del bloque 1, ahora convertida en columna permanente.
gen["es_renovable"] = gen["tecnologia"].isin(["hidro", "solar", "eolica"])

total = gen.groupby("es_renovable")["mwh"].sum()
print(total.round(0))
print("renovable", round(100 * total[True] / total.sum(), 1), "por ciento del total")

In [ ]:
#@title Ojo con estas dos cifras { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#fdecea;border-left:6px solid #c0392b;color:#7b241c"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Ojo con estas dos cifras</div><p>El factor de planta de la solar acá da 0,08 y en Chile ronda 0,19. Los datos del curso son sintéticos y ese número quedó bajo. Lo que sí es real es el orden, las térmicas muy por encima de la solar y la eólica.</p><p>La participación renovable acá da 46,0 por ciento. En el sistema real, contando toda la hidroelectricidad, la cifra de 2025 fue 65,6 por ciento.</p></div>"""))

In [ ]:
import numpy as np

# np.where recibe la condicion, el valor si se cumple y el valor si no.
gen["tamano"] = np.where(gen["potencia_mw"] >= 300, "grande", "chica")

print(gen.groupby("tamano")["central"].nunique())

In [ ]:
# bins son los cortes y labels los nombres. Hay un label menos que cortes.
gen["nivel_uso"] = pd.cut(gen["factor_planta"], bins=[-0.001, 0.3, 0.6, 1.0],
                          labels=["bajo", "medio", "alto"])

print(gen["nivel_uso"].value_counts().sort_index())

In [ ]:
# transform devuelve el total del grupo fila por fila, en vez de resumirlo.
gen["total_hora_region"] = gen.groupby(
    ["fecha", "hora", "region"])["mwh"].transform("sum")
gen["participacion"] = gen["mwh"] / gen["total_hora_region"]

print(len(gen), "filas antes y despues, el largo de la tabla no cambia")

In [ ]:
# Una hora concreta del Biobio, para ver cuanto aporto cada central.
muestra = gen[(gen["fecha"] == "2024-06-15") & (gen["hora"] == 20)
              & (gen["region"] == "Biobio")]

muestra[["central", "mwh", "total_hora_region", "participacion"]].round(3)

In [ ]:
# assign devuelve una copia con las columnas nuevas, sin tocar el original.
resumen = (gen
           .groupby("tecnologia")
           .agg(total=("mwh", "sum"), centrales=("central", "nunique"))
           .reset_index()
           .assign(promedio_por_central=lambda d: d["total"] / d["centrales"]))

resumen.round(0)

In [ ]:
# rename cambia nombres, drop elimina columnas y astype convierte tipos.
chica = resumen.rename(columns={"total": "total_mwh"}).drop(columns="centrales")
chica["total_mwh"] = chica["total_mwh"].astype(int)

chica.round(0)

In [ ]:
#@title Tu turno { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Tu turno</div><strong>Una columna de costo</strong>
<p>Crea una columna <code>costo_estimado</code> multiplicando <code>mwh</code> por el costo por MWh de cada tecnología. <code>map</code> traduce cada valor de una columna usando un diccionario. La primera línea ya está hecha, falta la multiplicación y el resumen por tecnología.</p></div>"""))

In [ ]:
# Tu turno
COSTO = {"hidro": 5, "solar": 2, "eolica": 3, "gas": 65, "carbon": 45, "diesel": 190}
gen["costo_unitario"] = gen["tecnologia"].map(COSTO)

# Falta multiplicar mwh por costo_unitario en una columna costo_estimado
# y resumir el total por tecnologia.
print(gen[["tecnologia", "mwh", "costo_unitario"]].head(3))

## 6. El precio contra la demanda

In [ ]:
import json

# La respuesta de la API del Lab 01, ahora como punto de partida.
cuerpo = json.load(open("precios_nudo.json"))
precios = pd.json_normalize(cuerpo["datos"])
precios["fecha"] = pd.to_datetime(precios["fecha"])

print(precios.shape)
precios.head(3)

In [ ]:
# Con una lista en on, el merge cruza por las tres columnas juntas.
enero = demanda[demanda["mes"] == 1]
mix = pd.merge(precios, enero, on=["fecha", "hora", "region"], how="inner")

print(mix.shape[0], "filas, que son las horas de enero por las siete regiones")
mix[["fecha", "hora", "region", "precio_usd_mwh", "mwh"]].head(3)

In [ ]:
# Correlacion entre precio y demanda sobre las 5.208 filas juntas.
# float() saca el envoltorio de numpy y deja el numero pelado.
print(round(float(mix["precio_usd_mwh"].corr(mix["mwh"])), 3))

In [ ]:
# La misma correlacion, calculada dentro de cada region por separado.
mix.groupby("region").apply(
    lambda x: x["precio_usd_mwh"].corr(x["mwh"]), include_groups=False
).round(3)

In [ ]:
#@title Tu turno { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#ffe9cf;border-left:6px solid #ff9933;color:#7a4700"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Tu turno</div><strong>El precio hora por hora</strong>
<p>Cambia el filtro para quedarte con la Metropolitana y agrupa por <code>hora</code>, para ver el precio promedio de cada hora del día. Mira a qué hora está el máximo.</p></div>"""))

In [ ]:
# Tu turno
# Cambia la region por Metropolitana y agrupa por hora en vez de por region.
seleccion = mix[mix["region"] == "Biobio"]

print(seleccion.groupby("region")["precio_usd_mwh"].mean().round(2))

In [ ]:
#@title Ojo con la hora de la punta { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#fdecea;border-left:6px solid #c0392b;color:#7b241c"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Ojo con la hora de la punta</div><p>En estos datos el precio máximo cae cerca de las 19 horas, porque así se construyó la demanda sintética.</p><p>En el sistema chileno real los máximos mensuales de demanda de 2025 cayeron todos entre las 11 y las 16 horas, y todos en día hábil. La forma de la curva de este laboratorio no reproduce la del sistema real.</p></div>"""))

## 7. La misma pelicula en Polars

In [ ]:
import polars as pl

# El mismo archivo y la misma union del principio, escritos en Polars.
gen_pl = pl.read_csv("generacion.csv", try_parse_dates=True).join(
    pl.read_csv("centrales.csv"), on="central", how="left")

print(gen_pl.shape)

In [ ]:
# pandas. Corchetes, y el nombre de la tabla repetido en cada condicion.
print("pandas", gen[(gen["tecnologia"] == "solar")
                    & (gen["region"] == "Antofagasta")].shape[0], "filas")

# Polars. filter con expresiones pl.col, sin repetir el nombre de la tabla.
print("polars", gen_pl.filter((pl.col("tecnologia") == "solar")
                              & (pl.col("region") == "Antofagasta")).shape[0], "filas")

In [ ]:
# pandas. agg con tuplas de columna y funcion.
print(gen.groupby("tecnologia").agg(total=("mwh", "sum"),
                                    promedio=("mwh", "mean")).round(1))

In [ ]:
# Polars. group_by con expresiones, y el nombre a la izquierda igual que antes.
gen_pl.group_by("tecnologia").agg(
    total=pl.col("mwh").sum(), promedio=pl.col("mwh").mean()).sort("total")

In [ ]:
# pandas crea la columna asignando a un nombre que no existe.
print(gen.assign(factor=gen["mwh"] / gen["potencia_mw"])["factor"].mean().round(3))

# Polars usa with_columns, y alias para ponerle nombre al resultado.
print(gen_pl.with_columns(
    (pl.col("mwh") / pl.col("potencia_mw")).alias("factor"))["factor"].mean())

In [ ]:
#@title Puntos clave { display-mode: "form" }
from IPython.display import HTML, display
display(HTML("""<div style="padding:14px 18px;border-radius:4px;line-height:1.55;font-size:15px;font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;background:#d8f0df;border-left:6px solid #28a745;color:#14532d"><div style="font-weight:700;font-size:16px;margin-bottom:6px">Puntos clave</div><ul>
<li>Elegir filas es pasarle a la tabla una columna de verdaderos y falsos, y elegir columnas es pasarle una lista de nombres</li>
<li>Con <code>dt</code> una fecha entrega el mes, el día de la semana y la hora, sin cortar texto</li>
<li><code>groupby</code> parte, calcula y junta, y con <code>agg</code> hace varios cálculos de una vez</li>
<li><code>pivot_table</code> y <code>melt</code> son la misma cifra puesta ancha o larga, se elige según quién la va a leer</li>
<li><code>transform</code> devuelve el resultado del grupo fila por fila, que es como se calcula una participación</li>
<li>Ninguna de estas operaciones cambia el archivo original, todas producen una tabla nueva</li>
</ul>
<p>En el Lab 03 vamos a tomar una tabla sucia y dejarla usable, que es el paso que falta antes de analizar.</p></div>"""))